# ShiftLog-Gym Smoke Test

Sanity-check the environment, collect baseline traces, and export simple metrics before RL.

In [ ]:
# Colab setup: clone repo + install it so `shiftlog_gym` is importable.
import os

REPO_URL = "https://github.com/Chirag0096/ShiftLog-Gym.git"
REPO_DIR = "ShiftLog-Gym"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!pip -q install -e .


In [ ]:
!pip -q install pandas matplotlib


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from shiftlog_gym.simulator import ShiftLogSimulator


In [ ]:
sim = ShiftLogSimulator()
obs = sim.reset(family='db_pool_exhaustion', variant_index=0)
print(obs)


In [ ]:
def run_scripted_episode(sim, family, variant_index=0):
    sim.reset(family=family, variant_index=variant_index)
    while not sim.done:
        incident = sim.active_incident
        if incident.required_memory_keys:
            sim.read_shift_log(' '.join(incident.relevant_memory_terms[:3]), limit=3)
        sim.inspect_service(incident.service)
        diag = next(iter(incident.diagnostics.keys()))
        sim.run_diagnostic(incident.service, diag)
        for _, fact in incident.golden_memory[:1]:
            sim.append_shift_log('fact', incident.incident_id, incident.service, fact, 0.95)
        sim.apply_mitigation(incident.service, incident.resolution)
        sim.resolve_incident(incident.incident_id, incident.resolution, incident.root_cause)
    return sim.get_state().model_dump()


rows = []
for family in ['db_pool_exhaustion', 'auth_timeout_cascade', 'memory_oom_signature', 'feature_flag_regression']:
    rows.append(run_scripted_episode(ShiftLogSimulator(), family))

df = pd.DataFrame(rows)
df


In [ ]:
ax = df.plot(x='scenario_family', y=['total_reward', 'recall_before_action_rate', 'linked_incident_success_rate'], kind='bar', figsize=(10, 4))
ax.set_title('ShiftLog-Gym Scripted Baseline')
ax.figure.tight_layout()
plt.show()


In [ ]:
artifact_dir = Path('smoke_artifacts')
artifact_dir.mkdir(exist_ok=True)
df.to_csv(artifact_dir / 'eval_summary.csv', index=False)
with open(artifact_dir / 'episodes.jsonl', 'w') as handle:
    for row in rows:
        handle.write(json.dumps(row) + '\n')
print('Wrote artifacts to', artifact_dir)
